In [1]:
!pip install pandas numpy statsmodels linearmodels scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.9/118.9 kB 5.2 MB/s eta 0:00:00


In [2]:
from google.colab import files
import pandas as pd

# Upload file
uploaded = files.upload()
df = pd.read_excel(list(uploaded.keys())[0])

df.head()

Saving climate-indicators.xlsx to climate-indicators.xlsx


,ADM0_NAME,ADM1_NAME,ADM2_CODE,ADM2_NAME,year,LST_Day_1km,NDVI,pdsi,precipitation,GPP,LSTN,NTL
0,Albania,Berat,3775,Berat,2000,301.123072,0.414728,-2.586312,763.180766,NaN,NaN,NaN
1,Albania,Berat,3776,Cukalat,2000,301.574553,0.431398,-2.553430,756.702377,NaN,NaN,NaN
2,Albania,Berat,3777,Kutalli,2000,301.008906,0.481690,-2.495369,784.134731,NaN,NaN,NaN
3,Albania,Berat,3778,Lumas,2000,300.083420,0.493094,-2.717357,826.618568,NaN,NaN,NaN
4,Albania,Berat,3779,Otllak,2000,300.709704,0.468100,-2.643816,768.996678,NaN,NaN,NaN


In [3]:
import numpy as np
from statsmodels.stats.outliers_influence import variance_inflation_factor

In [4]:
# combine ADM0 + ADM1 + ADM2 (bcs some countries have no name adm2 or adm1)
df["entity"] = df["ADM0_NAME"].astype(str) + " | " + df["ADM1_NAME"].astype(str) + " | " + df["ADM2_NAME"].astype(str)

# Sort by region + year
df = df.sort_values(["entity","year"])

# Climate variables
clim_vars = ["NDVI","GPP","LST_Day_1km","LSTN","pdsi","precipitation","NTL"]

# Create lagged variables (t-1)
for v in clim_vars:
    df[f"{v}_lag1"] = df.groupby("entity")[v].shift(1)

panel = df[df["year"].between(2012,2024)].copy()

In [6]:
clim_vars_early = ["NDVI","LST_Day_1km","pdsi","precipitation"]
clim_vars_late  = ["LSTN","NTL","GPP"]

df = df.copy()

# Early baseline (2000–2012) - this one is used for bands: NDVI, LST, PDSI, and PRECIP
baseline_early = (df[df["year"].between(2012,2019)] ######## edit: using same baseline for all here
                  .groupby("entity")[clim_vars_early]
                  .agg(["mean","std"]))
baseline_early.columns = ["_".join(col) for col in baseline_early.columns]

# Late baseline (2012–2019)  - this one is used for bands LSTN, NTL, GPP
baseline_late = (df[df["year"].between(2012,2019)]
                 .groupby("entity")[clim_vars_late]
                 .agg(["mean","std"]))
baseline_late.columns = ["_".join(col) for col in baseline_late.columns]

# Merge back
df = df.merge(baseline_early, left_on="entity", right_index=True, how="left")
df = df.merge(baseline_late, left_on="entity", right_index=True, how="left")

# Compute anomalies
for v in clim_vars_early:
    df[f"{v}_anom"] = (df[v] - df[f"{v}_mean"]) / df[f"{v}_std"]

for v in clim_vars_late:
    df[f"{v}_anom"] = (df[v] - df[f"{v}_mean"]) / df[f"{v}_std"]

# Create lagged anomalies
for v in clim_vars_early + clim_vars_late:
    df[f"{v}_anom_lag1"] = df.groupby("entity")[f"{v}_anom"].shift(1)


panel_anom = (df[df["year"].between(2012,2024)]
              .groupby(["entity","year"], as_index=False)
              .mean(numeric_only=True)
              .set_index(["entity","year"]))


In [7]:
clim_vars_early = ["NDVI","LST_Day_1km","pdsi","precipitation"]
clim_vars_late  = ["LSTN","NTL","GPP"]
all_vars = clim_vars_early + clim_vars_late

keep_cols = (["entity","year"] +
             [f"{v}_anom" for v in all_vars] +
             [f"{v}_anom_lag1" for v in all_vars])

panel_anom = (df[df["year"].between(2012,2024)][keep_cols]
              .groupby(["entity","year"], as_index=False)
              .mean(numeric_only=True)
              .set_index(["entity","year"])
              .sort_index())

assert not panel_anom.index.duplicated().any(), "Duplicate (entity,year) rows remain."


In [ ]:
panel_anom.reset_index().to_excel("panel_anom.xlsx", index=False)

In [ ]:
from google.colab import files
files.download("panel_anom.xlsx")